In [126]:
import numpy as np
import pandas as pd


In [127]:
online = pd.read_csv("eeg_lsl_raw_2back_-1580742246.csv")
online

,Time,Ch01,Ch02,Ch03,Ch04,Ch05,Ch06,LSL_timestamp
0,4.917181,16097.927734,-1099.062866,1.207939e+06,2.748139e+06,657448.875000,848193.875000,1.598252e+06
1,4.919369,16729.427734,-1040.322021,1.082270e+06,2.584417e+06,618281.000000,848193.875000,1.598252e+06
2,4.920556,17318.822266,-976.498718,9.535498e+05,2.446343e+06,585249.000000,848193.875000,1.598252e+06
3,4.923301,16209.460938,-909.301636,8.268294e+05,2.336490e+06,558968.312500,848193.875000,1.598252e+06
4,4.932822,16070.129883,-840.230286,7.059869e+05,2.255283e+06,539540.812500,848193.875000,1.598252e+06
...,...,...,...,...,...,...,...,...
11389,50.496724,12308.122070,-6.700386,4.489517e+01,1.262234e+02,29.641949,41.162376,1.598298e+06
11390,50.499842,14450.335938,-7.709904,5.944262e+01,1.141642e+02,26.756994,41.162376,1.598298e+06
11391,50.508713,15722.513672,-8.640556,7.465922e+01,1.044585e+02,24.435040,41.162376,1.598298e+06
11392,51.509405,0.000000,-9.501600,9.028042e+01,9.713584e+01,22.683218,41.162376,1.598299e+06


In [128]:
online = online.rename(columns={"Time": "Time", "Ch01": "Raw", "Ch02": "Theta Bandpass", "Ch03": "Power", "Ch04": "Moving Average", "Ch05": "Z-score", "Ch06": "Hold"})
online

,Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,LSL_timestamp
0,4.917181,16097.927734,-1099.062866,1.207939e+06,2.748139e+06,657448.875000,848193.875000,1.598252e+06
1,4.919369,16729.427734,-1040.322021,1.082270e+06,2.584417e+06,618281.000000,848193.875000,1.598252e+06
2,4.920556,17318.822266,-976.498718,9.535498e+05,2.446343e+06,585249.000000,848193.875000,1.598252e+06
3,4.923301,16209.460938,-909.301636,8.268294e+05,2.336490e+06,558968.312500,848193.875000,1.598252e+06
4,4.932822,16070.129883,-840.230286,7.059869e+05,2.255283e+06,539540.812500,848193.875000,1.598252e+06
...,...,...,...,...,...,...,...,...
11389,50.496724,12308.122070,-6.700386,4.489517e+01,1.262234e+02,29.641949,41.162376,1.598298e+06
11390,50.499842,14450.335938,-7.709904,5.944262e+01,1.141642e+02,26.756994,41.162376,1.598298e+06
11391,50.508713,15722.513672,-8.640556,7.465922e+01,1.044585e+02,24.435040,41.162376,1.598298e+06
11392,51.509405,0.000000,-9.501600,9.028042e+01,9.713584e+01,22.683218,41.162376,1.598299e+06


In [129]:
online.columns

Index(['Time', 'Raw', 'Theta Bandpass', 'Power', 'Moving Average', 'Z-score',
       'Hold', 'LSL_timestamp'],
      dtype='object')

In [130]:
merged_df = online

In [131]:
buffer = []
offline_z = []
last_theta_val = None

MU = 2.32
SIGMA = 4.18
MAD_THRESHOLD = 6
THETA_THRESHOLD_Z = 1.5
INITIAL_CUTOFF = 100.0
BUFFER_SIZE = 500
COOLDOWN_TIME = 7
sonication_enabled = 20

# Ch05 = index 4
channel = "Hold"

for i in range(len(merged_df)):

    theta_val = merged_df[channel].iloc[i]   # MUST match online
    if last_theta_val is not None and theta_val == last_theta_val:
        offline_z.append(last_theta_val) 
        continue
    # Warm-up
    if len(buffer) <= 200:
        if theta_val < INITIAL_CUTOFF:
            buffer.append(theta_val)
        offline_z.append(np.nan)
        continue

    # Rolling buffer
    if len(buffer) > BUFFER_SIZE:
        buffer.pop(0)

    arr = np.array(buffer)
    median = np.median(arr)
    mad = np.median(np.abs(arr - median)) + 1e-6

    # Artifact rejection
    z_art = abs(theta_val - median) / mad
    if z_art > MAD_THRESHOLD:
        offline_z.append(np.nan)
        continue

    # Clean sample
    buffer.append(theta_val)
    last_theta_val = theta_val

    # Z-score
    offline_z.append(theta_val)

merged_df["offline z-score"] = offline_z




In [132]:
LIFU = np.zeros(len(merged_df))
last_trigger_time = -999

for i in range(len(merged_df)):
    theta_z = merged_df["Hold"].iloc[i]
    now = merged_df["Time"].iloc[i]

    if now> sonication_enabled and theta_z > THETA_THRESHOLD_Z and theta_z < MAD_THRESHOLD and (now - last_trigger_time > COOLDOWN_TIME):
        LIFU[i] = 1.0
        last_trigger_time = now


In [133]:
merged_df["LIFU"] = LIFU
merged_df

,Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,LSL_timestamp,offline z-score,LIFU
0,4.917181,16097.927734,-1099.062866,1.207939e+06,2.748139e+06,657448.875000,848193.875000,1.598252e+06,NaN,0.0
1,4.919369,16729.427734,-1040.322021,1.082270e+06,2.584417e+06,618281.000000,848193.875000,1.598252e+06,NaN,0.0
2,4.920556,17318.822266,-976.498718,9.535498e+05,2.446343e+06,585249.000000,848193.875000,1.598252e+06,NaN,0.0
3,4.923301,16209.460938,-909.301636,8.268294e+05,2.336490e+06,558968.312500,848193.875000,1.598252e+06,NaN,0.0
4,4.932822,16070.129883,-840.230286,7.059869e+05,2.255283e+06,539540.812500,848193.875000,1.598252e+06,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...
11389,50.496724,12308.122070,-6.700386,4.489517e+01,1.262234e+02,29.641949,41.162376,1.598298e+06,NaN,0.0
11390,50.499842,14450.335938,-7.709904,5.944262e+01,1.141642e+02,26.756994,41.162376,1.598298e+06,NaN,0.0
11391,50.508713,15722.513672,-8.640556,7.465922e+01,1.044585e+02,24.435040,41.162376,1.598298e+06,NaN,0.0
11392,51.509405,0.000000,-9.501600,9.028042e+01,9.713584e+01,22.683218,41.162376,1.598299e+06,NaN,0.0


In [134]:
import math

nan_values = [x for x in offline_z if isinstance(x, float) and math.isnan(x)]
print(len(nan_values))

3160


In [136]:
mask_lifu = merged_df["LIFU"] == 1.0
lifu_on = merged_df[mask_lifu]
lifu_on["Time"] = lifu_on["Time"] + 0.7
lifu_on

C:\Users\jshin\AppData\Local\Temp\ipykernel_30276\4198628260.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lifu_on["Time"] = lifu_on["Time"] + 0.7


,Time,Raw,Theta Bandpass,Power,Moving Average,Z-score,Hold,LSL_timestamp,offline z-score,LIFU
3956,21.441740,16277.004883,4.454147,19.839428,9.146161,1.633053,1.633053,1.598268e+06,1.633053,1.0
5846,29.003722,13769.412109,1.842568,3.395058,10.248670,1.896811,1.896811,1.598276e+06,1.896811,1.0
8025,37.719534,13378.598633,-3.506470,12.295331,9.879150,1.808409,1.842245,1.598284e+06,1.842245,1.0
10345,47.004382,15727.710938,4.545255,20.659344,9.969485,1.830021,1.888382,1.598294e+06,1.888382,1.0
